In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq
import os

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14})

In [ ]:
from scipy.integrate import quad

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

In [ ]:
def load1dFig4(i):
    return np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD', 'data', str(i), 'data.tsv'))

def load2dFig4(i, num):
    tmp = np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD', 'data', str(i), 'data.tsv'))
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
from scipy.special import gamma as Gamma
# import matplotlib.pyplot as plt
from scipy.special import digamma, hyp2f1
from scipy.integrate import quad
from scipy.optimize import fsolve

#Following equations defined in PhysRevB.56.1848
def JsumComponent(t, R, Ec, T):
    #The infinite sum component of J(t) defined in above paper
    gamma = np.euler_gamma
    beta = 1.0/T
    x=beta*Ec/(2*R*np.pi**2)
    y=np.exp(-4*np.pi**2 * t/beta)
    return -(1/np.pi)*(2*gamma+digamma(-x)+digamma(x)+2*np.log(1-y)+(y/(1+x)) *hyp2f1(1,1+x,2+x,y) + (y/(1-x)) * hyp2f1(1, 1-x,2-x,y))

def J(t, R, Ec, T):
    #R=series resistance
    #assuming charge of electron, h = 1 (so Rk = e^2/h = 1)
    #kboltzmann = 1
    #Also assuming Ec = 1/2C (got from Ingold and Nazarov Chapter in Single Charge Tunneling Book, eq 66)
    wc = 2*Ec/R
    beta = 1.0/T

    return np.pi*R*((1-np.exp(-wc*abs(t)))*(np.tan(beta*wc/(4*np.pi))**(-1) -1j)- 4*np.pi*abs(t)/beta + JsumComponent(abs(t), R, Ec,T))

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EB(V, R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1))
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EBzero(R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    V = 0
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1))
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])


In [ ]:
def sineQ(Vp, A, w, phi, c, l):
    return A*np.sin(w*Vp+phi) + c + l*Vp

def KNtheory(tau0, T, Ec, Z, idcstart=-5e-9, idcstop=5e-9, npoints=51):
    idc = np.linspace(idcstart, idcstop, npoints)
    V = 1*idc*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV


def KNtheory2(tau0, T, Ec, Z):
    idc = np.linspace(-1.5e-8, 1.5e-8, 151)
    V = 1*idc[30:120]*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

def KNtheoryfull(tau0, V, T, Ec, Z):
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

@np.vectorize
def FurusakiMatveev(tau1, tau2, EcbykT, deltaVgbyDelta):
    Gamma = (1-tau1) + (1-tau2) - 2*np.sqrt((1-tau1)*(1-tau2))*np.cos(2*np.pi*deltaVgbyDelta)
    def integrand(x, Gamma, EcbykT):
        gamma = np.exp(0.5772)
        return (Gamma/np.cosh(x))**2 / ((x*np.pi**2/(gamma*EcbykT))**2 + Gamma**2)
    
    
    return 0.5*(1 - quad(integrand,0, np.inf, args = (Gamma,EcbykT))[0])
@np.vectorize
def QFurusakiMatveev(tau1, tau2, EcbykT):
    Gmax = FurusakiMatveev(tau1, tau2, EcbykT, 0)
    Gmin = FurusakiMatveev(tau1, tau2, EcbykT, 1/2)

    return (Gmax - Gmin)/(Gmax + Gmin)

In [ ]:
dat = load2dFig4(796, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5top = np.zeros(tau.shape)
start = 30
end = 52

start2 = start
start3 = 5

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
cbar = fig.colorbar(im, ax=ax[0], extend='max')
ax[0].set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5top = np.zeros((3,151))
pcovfig5top = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0top =  np.zeros((botqpc.shape[1]))
reftauinftop =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 40e-6, 0.99999*Z], [8e-6, 100e-6, 1.000001*Z])
# paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])

for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0top[i] = tau0
    if i>=10 and i<63:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        ax[1].plot(1e6*(idc[begin:end,i] - idc[75,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5top[:,j], pcovifig5top = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 90e-6, Z], bounds=paramBounds)
        pcovfig5top[:,j] = np.diag(pcovifig5top)
        print(poptfig5top[:,j])
        tauVfig5top[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5top[0,j], poptfig5top[1,j], poptfig5top[2,j])
        print(tauVfig5top[end-1,i])
        reftauinftop[i] = tauVfig5top[end-1,i]
        ax[1].plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5top[begin:end,i], ls='--', color='b', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
ax[0].set_ylabel('V$_{dc}$  ($\mu$V)')
ax[1].set_xlabel('V$_{dc}$  ($\mu$V)')
cbar.ax.set_ylabel('$\u03C4_{top}$')
ax[1].set_ylabel('$\u03C4_{top}$')
ax[1].grid(ls='--', lw=0.4)
# ax[1].set_ylim(0, 1.1)
fig.tight_layout()

# fig.savefig('figures/DCBtopQPConeRk.jpeg', dpi=300)
# fig.savefig('figures/DCBfitsKNmodel_oldZ1.jpeg', dpi=600)

In [ ]:
dat = load2dFig4(802, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5bot = np.zeros(tau.shape)
start = 30
end = 52

start2 = start
start3 = 5

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
cbar = fig.colorbar(im, ax=ax[0], extend='max')
ax[0].set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5bot = np.zeros((3,151))
pcovfig5bot = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0bot =  np.zeros((botqpc.shape[1]))
reftauinfbot =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 40e-6, 0.99999*Z], [8e-6, 100e-6, 1.000001*Z])
# paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])

for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0bot[i] = tau0
    if i>=20 and i<63:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        ax[1].plot(1e6*(idc[begin:end,i] - idc[75,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5bot[:,j], pcovifig5bot = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 95e-6, Z], bounds=paramBounds)
        pcovfig5bot[:,j] = np.diag(pcovifig5bot)
        print(poptfig5bot[:,j])
        tauVfig5bot[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5bot[0,j], poptfig5bot[1,j], poptfig5bot[2,j])
        print(tauVfig5bot[end-1,i])
        reftauinfbot[i] = tauVfig5bot[end-1,i]
        ax[1].plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5bot[begin:end,i], ls='--', color='b', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
ax[0].set_ylabel('V$_{dc}$  ($\mu$V)')
ax[1].set_xlabel('V$_{dc}$  ($\mu$V)')
cbar.ax.set_ylabel('$\u03C4_{top}$')
ax[1].set_ylabel('$\u03C4_{top}$')
ax[1].grid(ls='--', lw=0.4)
# ax[1].set_ylim(0, 1.1)
fig.tight_layout()

# fig.savefig('figures/DCBtopQPConeRk.jpeg', dpi=300)
# fig.savefig('figures/DCBfitsKNmodel_oldZ1.jpeg', dpi=600)

In [ ]:
fig5 = plt.figure(figsize=(12, 8),constrained_layout=True)
gs = fig5.add_gridspec(2, 2, width_ratios=(1,1))
# gs = GridSpec(3, 3, figure=fig1)
f5_ax1 = fig5.add_subplot(gs[:1, 0])
f5_ax2 = fig5.add_subplot(gs[:1, 1])
f5_ax3 = fig5.add_subplot(gs[1:2, 0])
f5_ax4 = fig5.add_subplot(gs[1:2, 1])

# inset_ax = fig3.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')

f5_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f5_ax1.transAxes)
f5_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f5_ax2.transAxes)
# f5_ax1.set_axis_off()
f5_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f5_ax3.transAxes)
f5_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f5_ax4.transAxes)

###################
dat = load2dFig4(802, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5bot = np.zeros(tau.shape)
# start = 30
# end = 52

# start2 = start
# start3 = 5

# im = f5_ax1.pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
# cbar = fig.colorbar(im, f5_ax1, extend='max')
# f5_ax1.set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5bot = np.zeros((3,151))
pcovfig5bot = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0bot =  np.zeros((botqpc.shape[1]))
reftauinfbot =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([1e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0bot[i] = tau0
    if i>=30 and i<51:
        j = i - 10
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        f5_ax1.plot(1e6*(idc[begin:end,i] - idc[76,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5bot[:,j], pcovifig5bot = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 100e-6, Z], bounds=paramBounds)
        pcovfig5bot[:,j] = np.diag(pcovifig5bot)
        print(poptfig5bot[:,j])
        tauVfig5bot[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5bot[0,j], poptfig5bot[1,j], poptfig5bot[2,j])
        # print(tauVfig5bot[end-1,i])
        reftauinfbot[i] = tauVfig5bot[end-1,i]
        f5_ax1.plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5bot[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
f5_ax1.set_ylabel('V$_{dc}$  ($\mu$V)')
f5_ax1.set_xlabel('V$_{dc}$  ($\mu$V)')
f5_ax1.set_ylabel('$\u03C4_{1}$')
f5_ax1.grid(ls='--', lw=0.4)
f5_ax1.axvline(idc[50,0]*25813/3*1e6, color='gray', ls='--')
f5_ax1.axvline(idc[101,0]*25813/3*1e6, color='gray', ls='--')

##############

start = 30
stop = 50

f5_ax2.plot(botqpc[0,start:stop], reftau0bot[start:stop], 'k', label='$V_{dc} = %1.1f \mu V$' %(idc[75,0]*25813*1e6/3))
f5_ax2.plot(botqpc[0,start:stop], tau[-6,start:stop], 'b', label='$V_{dc} = %1.1f \mu V$' %(idc[-6,0]*25813*1e6/3))
f5_ax2.plot(botqpc[0,start:stop], reftauinfbot[start:stop], 'm--', label='Eq. 9, $V_{dc} = %1.1f \mu V$'%(idc[-6,0]*25813*1e6/3))

f5_ax2.annotate("", xytext=(-3.04, 0.6), xy=(-3.04, 0.9),
            arrowprops=dict(arrowstyle="->"))

f5_ax2.grid(ls='--', lw=0.4)
f5_ax2.set_ylabel('$\u03C4_{1}$')
f5_ax2.set_xlabel('$V_g$')
f5_ax2.legend(loc='lower right')
f5_ax2.annotate("", xytext=(-3.005, 0.4), xy=(-3.005, 0.8),
            arrowprops=dict(arrowstyle="->"))

###############
dat = load2dFig4(796, 151)
topqpc = dat['1']
idc = dat['2']
vr = dat['3']*100/q2
vt = dat['5']*100/s1
vxx = dat['7']
vrdc = dat['9']
vtdc = dat['10']
vsdc = vrdc + vtdc

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1

tauVfig5top = np.zeros(tau.shape)
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5top = np.zeros((3,151))
pcovfig5top = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
rreftau0top =  np.zeros((topqpc.shape[1]))
reftauinftop =  np.zeros((topqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[76,i]
    reftau0top[i] = tau0
    if i>=12 and i<46:

        j = i-10
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        f5_ax3.plot(1e6*(idc[begin:end,i] - idc[76,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        # ax[1].plot(1e6*(vsdc[begin:end,i] - vsdc[75,i]), tau[begin:end,i],'r-', markersize=1, lw=1)
        lambdaKN = lambda tau0, T, Ec, Z : KNtheory(tau0, T, Ec, Z, idcstart=idc[50,0], idcstop=idc[100,0], npoints=51)
        poptfig5top[:,j], pcovifig5top = curve_fit(lambdaKN, tau0, tau[50:101,i], p0=[4.3e-6, 100e-6, Z], bounds=paramBounds)
        pcovfig5top[:,j] = np.diag(pcovifig5top)
        print(poptfig5top[:,j])
        tauVfig5top[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5top[0,j], poptfig5top[1,j], poptfig5top[2,j])
        print(tauVfig5top[end-1,i])
        reftauinftop[i] = tauVfig5top[end-1,i]
        f5_ax3.plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5top[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1


# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
f5_ax3.set_ylabel('V$_{dc}$  ($\mu$V)')
f5_ax3.set_xlabel('V$_{dc}$  ($\mu$V)')

f5_ax3.grid(ls='--', lw=0.4)
f5_ax3.axvline(idc[50,0]*25813/3*1e6, color='gray', ls='--')
f5_ax3.axvline(idc[100,0]*25813/3*1e6, color='gray', ls='--')

##############

start = 12
stop = 45

f5_ax4.plot(topqpc[0,start:stop], reftau0top[start:stop], 'k', label='$V_{dc} = %1.1f \mu V$' %(idc[75,0]*25813*1e6/3))
f5_ax4.plot(topqpc[0,start:stop], tau[-6,start:stop], 'b', label='$V_{dc} = %1.1f \mu V$' %(idc[-6,0]*25813*1e6/3))
f5_ax4.plot(topqpc[0,start:stop], reftauinftop[start:stop], 'm--',  label='Eq. 9, $V_{dc} = %1.1f \mu V$'%(idc[-6,0]*25813*1e6/3))

f5_ax4.annotate("", xytext=(-3.04, 0.6), xy=(-3.04, 0.9),
            arrowprops=dict(arrowstyle="->"))

f5_ax4.legend(loc='lower right')
f5_ax4.grid(ls='--', lw=0.4)
f5_ax4.set_ylabel('$\u03C4_{2}$')
f5_ax4.set_xlabel('$V_g$')
f5_ax4.annotate("", xytext=(-2.99, 0.2), xy=(-2.99, 0.75),
            arrowprops=dict(arrowstyle="->"))

f5_ax3.set_ylabel('$\\tilde{\u03C4}_2(V_{dc})$')
f5_ax1.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')
f5_ax2.set_ylabel('$\\tilde{\u03C4}_2(V_{dc})$')
f5_ax4.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')

fig5.savefig('DeviceB_DCB.pdf', dpi=300)
# fig5.savefig('DeviceB_DCB.eps', dpi=300)
# fig2.set_constrained_layout(False)
# fig3.tight_layout()

# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)


# # we don't want the layout to change at this point.
#fig1.tight_layout()

# skunk.display(svg)
# cairosvg.svg2pdf(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure5_Dec2025.pdf')
# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")